In [ ]:
import os
import glob
import pandas as pd

from judge import Judge

data_dir = "exp/attack"
judge_dir = "exp/judge"
os.mkdir(os.path.join(judge_dir, 'batch_input'), exist_ok=True)
os.mkdir(os.path.join(judge_dir, 'batch_output'), exist_ok=True)
os.mkdir(os.path.join(judge_dir, 'judge_result'), exist_ok=True)
llm_judge = Judge()

In [ ]:
batch_ids = {}
response_files = sorted(glob.glob(os.path.join(data_dir, '*', '*', '*.jsonl')))
for response_file in response_files:
    file_name = os.path.basename(response_file)
    input_path = os.path.join(judge_dir, 'batch_input', file_name)
    if os.path.exists(input_path):
        continue
    response_df = pd.read_json(response_file, lines=True)
    response_df = response_df[~response_df['behavior'].str.startswith('Function Misuse')]
    response_list = response_df[['behavior', 'adv_resp']].values.tolist()
    llm_judge.write_batch_jsonl(response_list, input_path)
    batch_id = llm_judge.create_batch_task(input_path)
    batch_ids[file_name] = batch_id
print(batch_ids)

In [ ]:
for file_name, batch_id in batch_ids.items():
    output_path = os.path.join(judge_dir, 'batch_output', file_name)
    status = llm_judge.check_batch_result(batch_id, output_path)
    batch_input_file = os.path.join(judge_dir, 'batch_input', file_name)
    if status == 'failed' and os.path.exists(batch_input_file):
        os.remove(batch_input_file)

In [ ]:
import re

def clean_behavior(s):
    return re.sub(r'(1|2|3|1[abc]|2[abc]|3[abc])$', '', s)

for response_file in response_files:
    file_name = os.path.basename(response_file)
    output_path = os.path.join(judge_dir, 'batch_output', file_name)
    if not os.path.exists(output_path):
        continue
    response_df = pd.read_json(response_file, lines=True)
    response_df = response_df[~response_df['behavior'].str.startswith('Function Misuse')]
    success_dict = llm_judge.parse_batch_result(output_path)
    result_df = response_df[['audio_type', 'behavior', 'strategy', 'prompt_type', 'adv_resp']].copy()
    result_df['behavior'] = result_df['behavior'].apply(clean_behavior)
    result_df['BMSR'] = [success_dict[str(i).zfill(3)] for i in range(len(result_df))]
    for audio_type in result_df['audio_type'].unique():
        sub_df = result_df[result_df['audio_type'] == audio_type]
        bmsr = sub_df.groupby('behavior')['PISR'].mean().round(2)
        print(' & '.join([audio_type] + [f'{asr:.2f}' for asr in bmsr]))
    result_df.to_json(os.path.join(judge_dir, 'judge_result', file_name), lines=True, orient='records', force_ascii=False)